# EU_09 – Practical Application of AI
## Parts: MNIST CNN, VGG16 Transfer Learning, and YOLOv8

In [ ]:
import os, numpy as np, tensorflow as tf, keras, matplotlib.pyplot as plt
from keras import layers
from keras.applications import VGG16

print('TensorFlow version:', tf.__version__)
print('GPUs available:', tf.config.list_physical_devices('GPU'))

# 1. MNIST CNN
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = np.expand_dims(x_train.astype('float32') / 255, -1)
x_test = np.expand_dims(x_test.astype('float32') / 255, -1)
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)

model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
print('Training MNIST CNN...')
model.fit(x_train, y_train, batch_size=256, epochs=5, validation_split=0.1, verbose=1)
print('\nTest accuracy:', model.evaluate(x_test, y_test, verbose=0)[1])

In [ ]:
# 2. VGG16 Transfer Learning
vgg16 = VGG16(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
for layer in vgg16.layers[:-4]: layer.trainable = False

transfer_model = keras.Sequential([
    vgg16, layers.Flatten(), layers.Dense(1024, activation='relu'),
    layers.Dropout(0.5), layers.Dense(3, activation='softmax')
])
transfer_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
print('VGG16 Model Summary:')
transfer_model.summary()

In [ ]:
# 3. YOLOv8
!pip install ultralytics -q
from ultralytics import YOLO
import cv2, requests
yolo = YOLO('yolov8s.pt')

def run_yolo(url):
    resp = requests.get(url, timeout=10)
    frame = cv2.imdecode(np.frombuffer(resp.content, np.uint8), cv2.IMREAD_COLOR)
    results = yolo(frame, verbose=False)
    for r in results:
        for box in r.boxes:
            if box.conf[0] > 0.4:
                x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

run_yolo('https://ultralytics.com/images/zidane.jpg')